# Model discriminability — can a trial even tell the competing models apart?

The model-selection arc (v0.21–v0.37) quantified how much the survival forecast depends on modeling
choices. This closes the loop: given two models' population OS curves, **what trial would it take to
distinguish them?** A divergence that needs tens of thousands of events is *practically unidentifiable
from the trial* — the model choice can only be assumed, not resolved by the data.

The required events for a logrank test is the Schoenfeld formula
$d = 4(z_{1-\alpha/2}+z_{1-\beta})^2 / (\ln \mathrm{HR})^2$, with HR the follow-up-horizon hazard
ratio between the curves. *Design/trial level only — a power calculation over published structures,
the v0.22/v0.31 identifiability family one level up at the model question; no individual quantity.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.discriminability import required_events, horizon_hazard_ratio, model_discriminability

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}

## The power core

HR = 0.5 needs ~65 events at 80% power (the standard logrank benchmark); as HR → 1 (smaller divergence)
the required events explode; identical curves are never distinguishable.

In [ ]:
for hr in (0.5, 0.7, 0.9, 0.97, 1.0):
    d = required_events(hr)
    print(f'HR={hr:.2f}  required events = {d:,.0f}' if np.isfinite(d) else f'HR={hr:.2f}  required events = inf')

## The finding: the resistance models are practically indistinguishable under week-8 OS

Under the default week-8 survival link, the pairs that diverge only in the regrowth *tail* (the
resistance mechanism, v0.24, and origin, v0.32) need an infeasible trial; the pairs that differ in
early *shrinkage* are easily told apart. The silent model-selection risk, quantified in events.

In [ ]:
md = model_discriminability(ds, context=ctx)
for p in sorted(md.pairs, key=lambda x: x['required_events']):
    a, b = p['record_a'].split('.')[-1], p['record_b'].split('.')[-1]
    d = p['required_events']
    flag = 'feasible' if d < 500 else ('large' if d < 3000 else 'INFEASIBLE')
    ds_ = f'{d:,.0f}' if np.isfinite(d) else 'inf'
    print(f'  HR={p["hazard_ratio"]:.2f}  events={ds_:>9}  {flag:<12} {a} vs {b}')
print(f'\n{md.n_indistinguishable}/{len(md.pairs)} pairs practically indistinguishable')

## The contrast: the metric choice's consequence IS detectable

The survival-METRIC choice (week-8 vs k_g for the same model) produces a large OS swing a small trial
detects — so the metric's consequence is identifiable, even though the model choice it is blind to is
not. The risk is concentrated exactly where the trial cannot look.

In [ ]:
t = np.linspace(0.0, 312.0, 625)
w = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t).os_curve
g = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t,
                   survival_link='survival_link.nsclc_os_growth_rate').os_curve
print('week-8 vs k_g (same model): events =', round(required_events(horizon_hazard_ratio(w, g))))
print('Claret vs two-population (week-8): events =',
      round([p['required_events'] for p in md.pairs
             if {p['record_a'].split('.')[-1], p['record_b'].split('.')[-1]} == {'tgi','two_population'}][0]))

## The takeaway

This reframes the project's load-bearing idea. The silent model-selection risks are silent *because*
they are practically unidentifiable from a surrogate-driven trial: distinguishing the resistance
mechanism or origin would take 10⁴–10⁵ events. You cannot resolve the model choice with data at a
feasible scale — only by assumption, with its tier and transportability attached. That is exactly the
uncertainty Onkos exists to make first-class. *Trial-level power calculation; no real trial designed,
no recommendation, no individual prediction.*